# Model optimization for transaction categorization (CPU only)
This notebook is a simplified, project-specific model optimization notebook for Chameleon. It compares eager PyTorch, compiled PyTorch, ONNX Runtime CPU, and ONNX Runtime dynamic quantization on a small synthetic transaction classifier.

In [ ]:
import os, time, json, random, tempfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import onnxruntime as ort
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)


## 1. Generate a synthetic transaction-like dataset

In [ ]:
cats = ["groceries","restaurants","transportation","rent","shopping","travel","utilities","subscriptions"]
merchant_map = {
    "WHOLE FOODS":"groceries",
    "TRADER JOES":"groceries",
    "UBER EATS":"restaurants",
    "STARBUCKS":"restaurants",
    "UBER TRIP":"transportation",
    "LYFT":"transportation",
    "LANDLORD LLC":"rent",
    "AMAZON":"shopping",
    "TARGET":"shopping",
    "DELTA AIR":"travel",
    "MARRIOTT":"travel",
    "DUKE ENERGY":"utilities",
    "NETFLIX":"subscriptions",
}

rows = []
for i in range(3000):
    merchant = random.choice(list(merchant_map))
    y = merchant_map[merchant]
    amount = {
        "groceries": np.random.normal(85, 20),
        "restaurants": np.random.normal(18, 6),
        "transportation": np.random.normal(24, 8),
        "rent": np.random.normal(1800, 80),
        "shopping": np.random.normal(60, 25),
        "travel": np.random.normal(350, 120),
        "utilities": np.random.normal(110, 30),
        "subscriptions": np.random.normal(14, 4),
    }[y]
    dow = random.randint(0, 6)
    credit = 1 if y in {"restaurants","shopping","travel","subscriptions","transportation"} else 0
    rows.append({
        "merchant_id": list(merchant_map).index(merchant),
        "amount": float(max(1, amount)),
        "dow": dow,
        "credit": credit,
        "label": cats.index(y),
    })

df = pd.DataFrame(rows)
X = df[["merchant_id","amount","dow","credit"]].astype(np.float32).values
y = df["label"].astype(np.int64).values

scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train)
X_test_t = torch.tensor(X_test)
y_test_t = torch.tensor(y_test)

X_train.shape, X_test.shape


## 2. Train a tiny surrogate PyTorch classifier

In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self, in_dim=4, hidden=32, out_dim=len(cats)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x):
        return self.net(x)

model = TinyClassifier()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(25):
    model.train()
    opt.zero_grad()
    logits = model(X_train_t)
    loss = loss_fn(logits, y_train_t)
    loss.backward()
    opt.step()

model.eval()
with torch.no_grad():
    logits = model(X_test_t)
    top1 = (logits.argmax(dim=1) == y_test_t).float().mean().item()
    top3 = (logits.topk(3, dim=1).indices == y_test_t.unsqueeze(1)).any(dim=1).float().mean().item()
print({"top1_accuracy": round(top1,4), "top3_hit_rate": round(top3,4)})


## 3. Export ONNX and create a dynamic-quantized ONNX model

In [ ]:
onnx_path = "models/tx_classifier.onnx"
quant_path = "models/tx_classifier_quant_dynamic.onnx"

dummy = torch.randn(1, 4)
torch.onnx.export(
    model,
    dummy,
    onnx_path,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)

from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(onnx_path, quant_path, weight_type=QuantType.QInt8)

print("Saved:", onnx_path, quant_path)


## 4. Benchmark helpers

In [ ]:
def benchmark_torch(torch_model, x_np, compiled=False, trials=200):
    m = torch_model
    if compiled:
        m = torch.compile(torch_model)
        # one warmup to trigger compilation
        _ = m(torch.tensor(x_np[:64]))
    x = torch.tensor(x_np)
    # warmup
    _ = m(x[:64])
    times = []
    for _ in range(trials):
        start = time.perf_counter()
        _ = m(x[:64])
        times.append((time.perf_counter() - start) * 1000.0)
    p50 = float(np.median(times))
    p95 = float(np.percentile(times, 95))
    throughput = (64 * trials) / (sum(times) / 1000.0)
    with torch.no_grad():
        logits = m(torch.tensor(x_np))
        top1 = (logits.argmax(dim=1).numpy() == y_test).mean()
        top3 = np.any(np.argsort(logits.numpy(), axis=1)[:, -3:] == y_test[:, None], axis=1).mean()
    return {"top1_accuracy": float(top1), "top3_hit_rate": float(top3), "p50_latency_ms": p50, "p95_latency_ms": p95, "throughput_items_per_sec": throughput}

def benchmark_onnx(path, x_np, trials=200):
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    input_name = sess.get_inputs()[0].name
    batch = x_np[:64]
    _ = sess.run(None, {input_name: batch})
    times = []
    for _ in range(trials):
        start = time.perf_counter()
        _ = sess.run(None, {input_name: batch})
        times.append((time.perf_counter() - start) * 1000.0)
    p50 = float(np.median(times))
    p95 = float(np.percentile(times, 95))
    throughput = (64 * trials) / (sum(times) / 1000.0)
    logits = sess.run(None, {input_name: x_np})[0]
    top1 = (np.argmax(logits, axis=1) == y_test).mean()
    top3 = np.any(np.argsort(logits, axis=1)[:, -3:] == y_test[:, None], axis=1).mean()
    return {"top1_accuracy": float(top1), "top3_hit_rate": float(top3), "p50_latency_ms": p50, "p95_latency_ms": p95, "throughput_items_per_sec": throughput}

def size_mb(path):
    return os.path.getsize(path) / 1e6


## 5. Run the four model-level variants

In [ ]:
results = []

eager_res = benchmark_torch(model, X_test, compiled=False)
eager_res.update({"option": "torch_eager_cpu", "artifact_size_mb": np.nan})
results.append(eager_res)

compiled_res = benchmark_torch(model, X_test, compiled=True)
compiled_res.update({"option": "torch_compiled_cpu", "artifact_size_mb": np.nan})
results.append(compiled_res)

onnx_res = benchmark_onnx(onnx_path, X_test)
onnx_res.update({"option": "onnx_cpu", "artifact_size_mb": size_mb(onnx_path)})
results.append(onnx_res)

quant_res = benchmark_onnx(quant_path, X_test)
quant_res.update({"option": "onnx_dynamic_quant_cpu", "artifact_size_mb": size_mb(quant_path)})
results.append(quant_res)

results_df = pd.DataFrame(results)
results_df.round(4)


## 6. Save summary for your serving-options table

In [ ]:
results_df.to_csv("results/model_optimization_summary.csv", index=False)
results_df.to_json("results/model_optimization_summary.json", orient="records", indent=2)
results_df
